# 7. Tarea elección abierta: Optuna + Random Forest

In [1]:
import optuna
import sklearn 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEED= 100499078

def objective(trial):
    rf = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500),
        max_depth=trial.suggest_int("max_depth", 3, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
        class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
        random_state=SEED,
        n_jobs=-1
    )

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", rf)
    ])

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv_inner,
        scoring="accuracy",   
        n_jobs=-1
    )

    return scores.mean()
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40)

print("Mejor accuracy CV:", study.best_value)
print("Mejores hiperparámetros:", study.best_params)

best_rf = RandomForestClassifier(
    **study.best_params,
    random_state=SEED,
    n_jobs=-1
)

best_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_rf)
])

best_pipe.fit(X_train, y_train)

y_pred = best_pipe.predict(X_test)

print("Accuracy en test:", accuracy_score(y_test, y_pred))
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
    

[I 2026-04-08 10:03:34,850] A new study created in memory with name: no-name-19efd69b-2ec0-45a7-a6e2-03a6d10b6b75
[W 2026-04-08 10:03:34,858] Trial 0 failed with parameters: {'n_estimators': 328, 'max_depth': 15, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_features': 'log2', 'bootstrap': True, 'class_weight': None} because of the following error: NameError("name 'Pipeline' is not defined").
Traceback (most recent call last):
  File "C:\Users\saram\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\saram\AppData\Local\Temp\ipykernel_61768\2454594374.py", line 22, in objective
    pipe = Pipeline(steps=[
           ^^^^^^^^
NameError: name 'Pipeline' is not defined
[W 2026-04-08 10:03:34,879] Trial 0 failed with value None.


NameError: name 'Pipeline' is not defined